# PhysIR — Stage 3: Weight Sweep (Kaggle)

**Purpose:** find the best phase × retinex weights. Stage 2 showed balanced tempering
kneecapped phase (combined 24.34 ≈ baseline, vs C's 24.94). So this sweep **restores phase
up/wide** and probes retinex around its solo 0.10. Blur is OFF for the grid (Stage 2 +
noedge showed it conflicts/adds nothing); a single low-value blur arm confirms that on the
grid winner.

**Grid (3×3 = 9) + blur arm (1) = 10 configs**, run one per session via `SELECT`.

| | retinex 0.05 | retinex 0.10 | retinex 0.18 |
|--|--|--|--|
| **phase 0.05** | ph05_re05 | ph05_re10 | ph05_re18 |
| **phase 0.08** | ph08_re05 | ph08_re10 | ph08_re18 |
| **phase 0.12** | ph12_re05 | ph12_re10 | ph12_re18 |

Config 10 = **blur arm**: grid winner's weights + blur=0.02 (fill after grid completes).

**Ranking metric:** LOLBlur val PSNR @ 30 ep. Gen eval (LOLv2) + perceptual are DROPPED —
they don't inform weight choice (reinstated at Stage 4/5). References: A=24.24, C=24.94,
Stage2=24.34.

**Environment (matches the Stage 2 run):** code + data are Kaggle *input datasets* (read-only,
no git clone); cwd stays `/kaggle/working` so `.metrics` lands there.

**Accelerator:** GPU T4 × 2.  **Quota:** ~3.5–4 hr/config → split across 2 weeks.

> ⚠️ **Re-upload the `physir-code` dataset with the fixed `train.py`** (phys-loss resume
> restore) before running the blur arm or any config that resumes — otherwise a resumed
> run silently resets the blur PSF.

## 1. Install dependencies

In [ ]:
!pip install -q lpips pytorch-msssim einops ptflops pyiqa

## 2. Paths — edit before running

In [ ]:
import os

# ── EDIT THESE (Kaggle input datasets — read-only) ────────────────────────────
CODE_DIR     = '/kaggle/input/datasets/jeffreyboahen/physir-code'      # repo as a dataset
LOLBLUR_ROOT = '/kaggle/input/datasets/jeffreyboahen/lolblur-dataset'  # LOLBlur
CHECKPOINT_DIR = '/kaggle/working'
# ──────────────────────────────────────────────────────────────────────────────

# cwd stays /kaggle/working (do NOT chdir into the read-only input) so that
# train.py writes .metrics under /kaggle/working/.metrics.
WORKING_DIR = '/kaggle/working'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
assert os.path.isdir(CODE_DIR), f'CODE_DIR not found: {CODE_DIR}'
assert os.path.isfile(os.path.join(CODE_DIR, 'train.py')), 'train.py not in CODE_DIR'
print(f'Code:        {CODE_DIR}')
print(f'Dataset:     {LOLBLUR_ROOT}')
print(f'Checkpoints: {CHECKPOINT_DIR}')
print(f'cwd:         {os.getcwd()}  (.metrics → {WORKING_DIR}/.metrics)')

## 3. Train/val split (same 90/10, seed 42 — identical to pilots/Stage 2)

In [ ]:
import random
from pathlib import Path

SUBS       = ['low_blur_noise', 'high_sharp_scaled']
TRAIN_SRC  = Path(LOLBLUR_ROOT) / 'train'
SPLIT_ROOT = Path(WORKING_DIR) / 'LOLBlur_split'

random.seed(42)
clips = sorted([d.name for d in (TRAIN_SRC / 'low_blur_noise').iterdir() if d.is_dir()])
n_val = max(1, int(len(clips) * 0.1))
val_clips   = set(random.sample(clips, n_val))
train_clips = [c for c in clips if c not in val_clips]
print(f'Clips — train: {len(train_clips)}, val: {len(val_clips)}')

for split_name, clip_list in [('physir_train', train_clips), ('physir_val', val_clips)]:
    for sub in SUBS:
        dst_base = SPLIT_ROOT / split_name / sub
        dst_base.mkdir(parents=True, exist_ok=True)
        for clip in clip_list:
            src_path = (TRAIN_SRC / sub / clip).resolve()
            dst_path = dst_base / clip
            if not dst_path.exists():
                dst_path.symlink_to(src_path)

PHYSIR_TRAIN = str(SPLIT_ROOT / 'physir_train')
PHYSIR_VAL   = str(SPLIT_ROOT / 'physir_val')
print(f'  train: {PHYSIR_TRAIN}')
print(f'  val:   {PHYSIR_VAL}')

## 4. Sweep grid — pick ONE config per session via `SELECT`

Run configs 0–8 (the grid) in any order across sessions. After the grid completes, run the
leaderboard cell, then set the blur arm (config 9) to the winner's weights and run it.

In [ ]:
# ── SELECT which config to run this session (0–9) ─────────────────────────────
SELECT = 0
# ──────────────────────────────────────────────────────────────────────────────

# name suffix = weight × 100 (05→0.05, 08→0.08, 10→0.10, 12→0.12, 18→0.18)
SWEEP = [
    # ── 3×3 grid (blur OFF) ──
    {'name': 'sweep_ph05_re05', 'phase': 0.05, 'retinex': 0.05, 'blur': 0.0},
    {'name': 'sweep_ph05_re10', 'phase': 0.05, 'retinex': 0.10, 'blur': 0.0},
    {'name': 'sweep_ph05_re18', 'phase': 0.05, 'retinex': 0.18, 'blur': 0.0},
    {'name': 'sweep_ph08_re05', 'phase': 0.08, 'retinex': 0.05, 'blur': 0.0},
    {'name': 'sweep_ph08_re10', 'phase': 0.08, 'retinex': 0.10, 'blur': 0.0},
    {'name': 'sweep_ph08_re18', 'phase': 0.08, 'retinex': 0.18, 'blur': 0.0},
    {'name': 'sweep_ph12_re05', 'phase': 0.12, 'retinex': 0.05, 'blur': 0.0},
    {'name': 'sweep_ph12_re10', 'phase': 0.12, 'retinex': 0.10, 'blur': 0.0},
    {'name': 'sweep_ph12_re18', 'phase': 0.12, 'retinex': 0.18, 'blur': 0.0},
    # ── blur arm (config 9): EDIT phase/retinex to the grid winner before running ──
    {'name': 'sweep_blurarm',   'phase': 0.08, 'retinex': 0.10, 'blur': 0.02},
]

SEL = SWEEP[SELECT]
print(f'Selected config {SELECT}: {SEL["name"]}')
print(f'  phase={SEL["phase"]}  retinex={SEL["retinex"]}  blur={SEL["blur"]}')
if SEL['name'] == 'sweep_blurarm':
    print('  ⚠️  BLUR ARM — confirm phase/retinex match the grid winner (see leaderboard),')
    print('      and that CODE_DIR has the resume-fixed train.py.')

## 5. Config for the selected sweep point

In [ ]:
import yaml

run_name = SEL['name']
blur_on  = SEL['blur'] > 0

cfg = {
    'datasets': {
        'name': 'LOLBlur',
        'train': {
            'low_path':   f'{PHYSIR_TRAIN}/low_blur_noise',
            'high_path':  f'{PHYSIR_TRAIN}/high_sharp_scaled',
            'crop_size':  256, 'batch_size': 4, 'n_workers': 4, 'use_flips': True,
        },
        'val': {
            'test_path': PHYSIR_VAL, 'batch_size_test': 1,
            'verbose': False, 'n_workers': 4,
        },
    },
    'network': {
        'name': 'DarkIR', 'img_channels': 3, 'width': 32,
        'middle_blk_num_enc': 2, 'middle_blk_num_dec': 2,
        'enc_blk_nums': [1, 2, 3], 'dec_blk_nums': [3, 1, 1],
        'dilations': [1, 4, 9], 'extra_depth_wise': True,
    },
    'save': {
        'pretrained': None,
        'path': f'{CHECKPOINT_DIR}/{run_name}_latest.pt',
        'best': f'{CHECKPOINT_DIR}/{run_name}_best.pt',
        'save_freq': 5, 'eval_freq': 5,
        'perceptual_eval_epochs': [],   # DROPPED for sweep — not a ranking metric
    },
    'pixel_criterion': 'l1', 'perceptual': False,
    'edge': True, 'edge_weight': 0.05, 'edge_criterion': 'l2', 'edge_reduction': 'mean',
    'frequency': False,
    'enhance': True, 'enhance_weight': 0.01,
    'enhance_criterion': 'l1', 'enhance_reduction': 'mean',
    'physics_losses': {
        'enabled': True,
        'phase2_start': 9999, 'phase3_start': 9999, 'phase4_start': 1,  # all gates from ep1
        'retinex': True,  'retinex_weight': SEL['retinex'],
        'blur':    blur_on, 'blur_weight':  SEL['blur'],
        'phase':   True,  'phase_weight':   SEL['phase'],
    },
    'train': {
        'epochs': 30, 'lr_initial': 0.00028, 'weight_decay': 0.001,
        'lr_scheme': 'CosineAnnealing', 'eta_min': 1e-6, 'grad_clip': 1.0,
    },
    # NOTE: no 'datasets_gen' block → LOLv2 gen eval OFF for the sweep.
    'wandb': {'init': False, 'name': run_name},
    'Resize': False,
}

cfg_path = f'{WORKING_DIR}/{run_name}.yml'
with open(cfg_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

pl = cfg['physics_losses']
print(f'Config written: {cfg_path}  (run_name={run_name})')
print(f'  phase={pl["phase_weight"]}  retinex={pl["retinex_weight"]}  '
      f'blur={pl["blur_weight"]} (on={pl["blur"]})')
print(f'  30 ep · LOLBlur val every 5 ep · gen+perceptual OFF')

## 6. Resume detection (for the selected config)

In [ ]:
latest_ckpt = f'{CHECKPOINT_DIR}/{run_name}_latest.pt'
if os.path.isfile(latest_ckpt):
    size_mb = os.path.getsize(latest_ckpt) / 1e6
    import torch as _t
    _ckpt = _t.load(latest_ckpt, map_location='cpu', weights_only=False)
    print(f'✅ Checkpoint found ({size_mb:.1f} MB) — resuming {run_name} from epoch {_ckpt.get("epoch", "?")}')
    del _ckpt
else:
    print(f'No checkpoint for {run_name} — training from scratch.')

## 7. Train the selected config

In [ ]:
!torchrun --nproc_per_node=2 {CODE_DIR}/train.py -p {cfg_path}

## 8. Leaderboard — rank all completed sweep configs

Run any time to see current standings. Reads every `.metrics/sweep_*` dir under WORKING_DIR.

In [ ]:
import pandas as pd
from pathlib import Path

metrics_root = Path(WORKING_DIR) / '.metrics'
rows = []
for d in sorted(metrics_root.glob('sweep_*')):
    log = d / 'training_log.csv'
    if not log.exists():
        continue
    df = pd.read_csv(log)
    df.replace('', float('nan'), inplace=True)
    val = df.dropna(subset=['val_psnr']) if 'val_psnr' in df.columns else df.iloc[0:0]
    if not len(val):
        rows.append({'config': d.name, 'best_psnr': None, 'epochs': int(df['epoch'].max())})
        continue
    best = val.loc[val['val_psnr'].idxmax()]
    rows.append({
        'config':    d.name,
        'best_psnr': round(float(best['val_psnr']), 4),
        'best_ssim': round(float(best['val_ssim']), 4) if pd.notna(best.get('val_ssim')) else None,
        'best_lpips':round(float(best['val_lpips']), 4) if pd.notna(best.get('val_lpips')) else None,
        'at_epoch':  int(best['epoch']),
        'epochs':    int(df['epoch'].max()),
    })

if not rows:
    print('No sweep runs found yet under', metrics_root)
else:
    board = pd.DataFrame(rows).sort_values('best_psnr', ascending=False, na_position='last')
    print('=== Stage 3 Sweep Leaderboard (LOLBlur val PSNR @ ≤30 ep) ===')
    print('  References:  A baseline=24.24   C phase-only=24.94   Stage2 combined=24.34\n')
    print(board.to_string(index=False))
    done = board[board['epochs'] >= 30]
    if len(done):
        w = done.iloc[0]
        print(f'\n  Leader (≥30 ep): {w["config"]}  →  {w["best_psnr"]} dB')
        print(f'  vs C (24.94): {w["best_psnr"] - 24.94:+.4f} dB    vs Stage2 (24.34): {w["best_psnr"] - 24.34:+.4f} dB')
        print('  → set the blur arm (config 9) to this config\'s phase/retinex, then run it.')